# Clase 137 — LSTM y GRU

**LSTM** (1997) y **GRU** (2014) son celdas recurrentes con **gates** que resuelven
el vanishing gradient de `SimpleRNN`: la información fluye por la **cell state** `c_t`
casi sin atenuación, y los gates aprenden qué **olvidar**, **recordar** y **emitir**.

Requiere: `tensorflow` / `keras` (≥ 3.0), `numpy`. Se ejecuta en Colab con GPU.

## 1. Datos con dependencia long-range

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
# El target depende del PRIMER y del ÚLTIMO paso -> requiere memoria a 100 pasos
T, F = 100, 1
r = np.random.default_rng(42)
X = r.standard_normal((2000, T, F)).astype("float32")
y = ((X[:, 0, 0] + X[:, -1, 0]) > 0).astype("float32")
print("X:", X.shape, "| balance de clases:", round(float(y.mean()), 3))

## 2. LSTM: 3 gates (forget, input, output) + cell state

In [ ]:
lstm = keras.Sequential([
    keras.Input(shape=(T, F)),
    layers.LSTM(32),                              # forget/input/output + c_t
    layers.Dense(1, activation="sigmoid"),
])
lstm.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
lstm.summary()

## 3. GRU: update + reset gate (más simple, menos parámetros)

In [ ]:
gru = keras.Sequential([
    keras.Input(shape=(T, F)),
    layers.GRU(32),                               # combina forget+input en 'update'
    layers.Dense(1, activation="sigmoid"),
])
gru.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
gru.summary()

## 4. Conteo de parámetros: SimpleRNN vs GRU vs LSTM

In [ ]:
u = 32
for nombre, capa in [("SimpleRNN", layers.SimpleRNN(u)),
                     ("GRU", layers.GRU(u)),
                     ("LSTM", layers.LSTM(u))]:
    capa.build((None, T, F))
    print(f"{nombre:10s}: {capa.count_params():>5d} params")
# GRU ~3x SimpleRNN, LSTM ~4x SimpleRNN (por sus gates).

## 5. `return_state`: recuperar hidden state `h` y cell state `c`

In [ ]:
inp = keras.Input(shape=(T, F))
salida, h, c = layers.LSTM(16, return_state=True)(inp)   # h = memoria corta, c = memoria larga
print("output:", salida.shape, "| estado h:", h.shape, "| estado c:", c.shape)
# Útil para encoder-decoder: el estado final del encoder inicializa el decoder.

## 6. `Bidirectional` para tareas no causales

In [ ]:
bi = keras.Sequential([
    keras.Input(shape=(T, F)),
    layers.Bidirectional(layers.LSTM(32)),   # forward + backward, concatena -> 64
    layers.Dense(1, activation="sigmoid"),
])
bi.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
bi.summary()
# OJO: Bidirectional ve el futuro -> NO usar en forecasting causal.

## Ejercicios

1. **LSTM vs SimpleRNN**: en la tarea long-range de 100 pasos, compará accuracy; la
   LSTM debería ganar claramente.
2. **GRU vs LSTM**: misma tarea, compará parámetros y accuracy (diferencia típica < 0.5 pp).
3. **Stacked**: apilá 2-3 LSTM con `return_sequences=True` en las primeras.
4. **Bidirectional**: para sentimiento, compará `LSTM` vs `Bidirectional(LSTM)`.

## Conclusiones

- **LSTM** = 3 gates + `c_t`; **GRU** = 2 gates + 1 estado (más liviana, casi igual de buena).
- Los gates permiten que el gradiente fluya por `c_t` sin atenuarse → resuelven el vanishing.
- `return_state=True` devuelve `(output, h, c)` para inicializar otro decoder.
- `Bidirectional` mejora tareas **no causales** (clasificación, NER); prohibido en forecasting.
- Empezá con LSTM; usá GRU por eficiencia. Con `T > 100`, incluso LSTM sufre → Transformer.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README. Como TensorFlow/PyTorch no están instalados en este entorno, las celdas de deep learning se validan por API (se ejecutan en Colab con GPU); las de **NumPy puro** son autónomas y traen `assert` para verificarse aquí mismo.

### Ejercicio 1 — LSTM vs SimpleRNN en tarea long-range

El target depende del primer y del último paso (memoria a 100 pasos). La LSTM, con su `cell state`, gana claramente; la SimpleRNN se queda en ~azar por vanishing.

In [ ]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

T, F = 100, 1
r = np.random.default_rng(42)
X = r.standard_normal((2000, T, F)).astype("float32")
y = ((X[:, 0, 0] + X[:, -1, 0]) > 0).astype("float32")   # depende de t=0 y t=99

def clf(celda):
    m = keras.Sequential([keras.Input(shape=(T, F)), celda(32),
                          layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"]); return m

rnn, lstm = clf(layers.SimpleRNN), clf(layers.LSTM)
# rnn.fit(X, y, epochs=5); lstm.fit(X, y, epochs=5)  -> LSTM ~0.9, SimpleRNN ~0.5
print("SimpleRNN vs LSTM en dependencia a 100 pasos: la LSTM gana por el cell state")

### Ejercicio 2 — GRU vs LSTM: parámetros y accuracy

La GRU fusiona forget+input en un solo `update gate`: ~25% menos parámetros, accuracy casi igual (< 0.5 pp de diferencia típica).

In [ ]:
from tensorflow.keras import layers

T, F, u = 100, 1, 32
info = {}
for nombre, celda in [("SimpleRNN", layers.SimpleRNN(u)), ("GRU", layers.GRU(u)),
                      ("LSTM", layers.LSTM(u))]:
    celda.build((None, T, F))
    info[nombre] = celda.count_params()
    print(f"{nombre:10s}: {info[nombre]:>5d} params")
assert info["GRU"] < info["LSTM"]            # GRU más liviana

### Ejercicio 3 — Stacked LSTM (2-3 capas)

`return_sequences=True` en todas menos la última para encadenar las salidas.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

T, F = 100, 1
apilada = keras.Sequential([
    keras.Input(shape=(T, F)),
    layers.LSTM(32, return_sequences=True),   # devuelve la secuencia
    layers.LSTM(32, return_sequences=True),
    layers.LSTM(32),                          # solo el último estado
    layers.Dense(1, activation="sigmoid"),
])
print("stacked 3xLSTM -> params:", apilada.count_params())

### Ejercicio 4 — Bidirectional para tareas no causales

`Bidirectional(LSTM)` corre forward + backward y concatena (dobla las unidades). Mejora clasificación de sentimiento; **prohibido** en forecasting (ve el futuro).

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

T, F = 100, 1
uni = keras.Sequential([keras.Input(shape=(T, F)), layers.LSTM(64),
                        layers.Dense(1, activation="sigmoid")])
bi = keras.Sequential([keras.Input(shape=(T, F)), layers.Bidirectional(layers.LSTM(64)),
                       layers.Dense(1, activation="sigmoid")])
print("LSTM:", uni.count_params(), "| Bidirectional(LSTM):", bi.count_params(),
      "-> ~2x params, ve contexto en ambos sentidos")

### Ejercicio 5 — cuDNN check: `recurrent_dropout` desactiva el kernel rápido

El kernel cuDNN acelera mucho la LSTM en GPU, pero solo con la config por defecto. `recurrent_dropout>0` (o activaciones no estándar) fuerza el fallback lento.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
import time

T, F = 100, 1
rapida = keras.Sequential([keras.Input(shape=(T, F)), layers.LSTM(64)])              # usa cuDNN
lenta = keras.Sequential([keras.Input(shape=(T, F)), layers.LSTM(64, recurrent_dropout=0.2)])  # sin cuDNN
# X = ...; t0 = time.time(); rapida.predict(X); t_rapida = time.time() - t0  (en GPU: mucho menor)
print("LSTM por defecto -> cuDNN (rápida); recurrent_dropout>0 -> fallback lento")